# E25 — official visual benchmark
Select a GPU runtime and Run all. SCOPE='pilot' runs HandReach / MAE-B / seed 0, SCOPE='full' runs all 24 Table 17 cells. Both score automatically. Full is expensive. No benchmark result yet; eight local fixture tests passed. No ACTUS reproduction or SOTA claim.

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess
SCOPE = 'pilot' # @param ["pilot", "full"]
assert SCOPE in ('pilot', 'full')
ROOT = Path('/content/e25_work_v2')
ROOT.mkdir(parents=True, exist_ok=True)
UP = ROOT/'counterfactual-WM'
BUNDLE = ROOT/'e25'
BUNDLE.mkdir(exist_ok=True)
PIN = '5c84bba92b2ff19d41175070a61238fa1396d10d'
ENV = dict(os.environ, MUJOCO_GL='egl', PYOPENGL_PLATFORM='egl', SDL_VIDEODRIVER='dummy', WANDB_MODE='disabled')
def shell(*args, cwd=ROOT, capture=False):
    args = list(map(str,args))
    print('>', ' '.join(args), flush=True)
    return subprocess.run(args,cwd=cwd,env=ENV,check=True,text=True,stdout=subprocess.PIPE if capture else None).stdout
assert shutil.which('nvidia-smi'), 'Select GPU runtime first'
gpu = shell('nvidia-smi','--query-gpu=name,driver_version,memory.total','--format=csv,noheader',capture=True)
print(gpu)
for line in gpu.strip().splitlines():
    assert int(line.split(',')[1].strip().split('.')[0]) >= 580, 'Pinned CUDA 13 needs R580+'
print('Free GiB:', shutil.disk_usage(ROOT).free/2**30)
if not UP.exists(): shell('git','clone','https://github.com/galilai-group/counterfactual-WM',UP)
shell('git','checkout','--detach',PIN,cwd=UP)
assert shell('git','rev-parse','HEAD',cwd=UP,capture=True).strip()==PIN

## Install the published environment
Separate Python 3.11 environment. No silent version substitution.

In [ ]:
if not shutil.which('uv'):
    shell(sys.executable,'-m','pip','install','uv')
UV = shutil.which('uv')
assert UV
VENV = ROOT/'venv'
if not (VENV/'bin/python').exists():
    shell(UV,'python','install','3.11')
    shell(UV,'venv','--python','3.11','--seed',VENV)
PY = VENV/'bin/python'
assert shell(PY,'-c','import sys; print(sys.version_info[:2])',capture=True).strip() == '(3, 11)'
shell(PY,'-m','pip','install','-r',UP/'requirements.txt',cwd=UP)
shell(PY,'-m','pip','install','--no-deps','-r',UP/'requirements-nodeps.txt',cwd=UP)
shell(PY,'-m','pip','install','--no-deps','-e',UP,cwd=UP)
shell(PY,'-c',"import torch, mujoco, gymnasium, stable_pretraining; assert torch.cuda.is_available(); print(torch.__version__,torch.version.cuda,torch.cuda.get_device_name(0)); print((torch.ones(4,device='cuda')*2).sum().item())",cwd=UP)

## Embedded runnable sources
Same code as the repository folder. No external bundle upload required.

In [ ]:
import json
sources = json.loads("{\"e25.py\":\"\\\"\\\"\\\"E25: regularized actuator/output disturbance inference around a frozen CWM.\\nOnly the correction changes; upstream CEM, encoder, predictor and readout stay fixed.\\nAll output coordinates are standardized with the upstream evaluation scale.\\n\\\"\\\"\\\"\\nimport torch\\n\\nARMS = ('trained_predictor', 'counterfactual', 'scalar', 'bias', 'actuator', 'joint')\\n\\nclass Correction:\\n    def __init__(self, model, sigma, limit, noise, alpha, arm):\\n        if arm not in ARMS:\\n            raise ValueError(arm)\\n        self.m, self.arm, self.limit = model, arm, limit\\n        self.sigma = torch.as_tensor(sigma, device=model.device, dtype=torch.float32)\\n        self.noise = torch.as_tensor(noise, device=model.device, dtype=torch.float32)\\n        if self.noise.shape != self.sigma.shape or not torch.all(self.noise > 0) or not torch.all(self.sigma > 0):\\n            raise ValueError('positive per-coordinate sigma and nominal residual standard deviation required')\\n        self.alpha = float(alpha)\\n        self.a, self.d = model.a, len(self.sigma)\\n        # Dimensionless gain and offset; output bias in standardized coordinates.\\n        prior = torch.cat([self.sigma.new_full((2*self.a,), .25), self.sigma.new_full((self.d,), .25)])\\n        mask = {'bias': [False]*(2*self.a)+[True]*self.d,\\n                'actuator': [True]*(2*self.a)+[False]*self.d}.get(arm, [True]*(2*self.a+self.d))\\n        self.mask = torch.tensor(mask, device=model.device)\\n        self.prec = torch.diag(prior[self.mask].reciprocal().square())\\n        self.rhs = torch.zeros_like(prior[self.mask])\\n        self.last = torch.zeros_like(self.sigma)\\n        self.raw_last = torch.zeros_like(self.sigma)\\n\\n    def mean(self, z, actions):\\n        return self.m.decode(self.m.predict(z, actions))\\n\\n    @torch.no_grad()\\n    def observe(self, z, action, decoded_outcome):\\n        action = action.reshape(-1)\\n        pred = self.mean(z, action[None])[0]\\n        self.raw_last = decoded_outcome - pred\\n        self.last = self.raw_last / self.sigma\\n        if self.arm not in ('bias', 'actuator', 'joint'):\\n            return\\n        h = .01 * self.limit\\n        eye = torch.eye(self.a, device=self.m.device, dtype=action.dtype) * h\\n        plus = (action[None]+eye).clamp(-self.limit, self.limit)\\n        minus = (action[None]-eye).clamp(-self.limit, self.limit)\\n        denom = (plus-minus).diagonal().clamp_min(1e-8)\\n        # Derivative of decoded frozen predictor, never a simulator derivative.\\n        j = ((self.mean(z, plus)-self.mean(z, minus))/denom[:,None]).T / self.sigma[:,None]\\n        design = torch.cat([j*action, j*self.limit, torch.eye(self.d, device=self.m.device)], dim=1)[:,self.mask]\\n        design = design / self.noise[:,None]\\n        target = self.last / self.noise\\n        self.prec += design.T @ design\\n        self.rhs += design.T @ target\\n\\n    @torch.no_grad()\\n    def parameters(self):\\n        p = self.sigma.new_zeros(2*self.a+self.d)\\n        p[self.mask] = torch.linalg.solve(self.prec, self.rhs)\\n        return p\\n\\n    @torch.no_grad()\\n    def predict(self, z, actions):\\n        if self.arm == 'trained_predictor':\\n            return self.mean(z, actions)\\n        if self.arm in ('counterfactual', 'scalar'):\\n            return self.mean(z, actions) + self.raw_last*(1 if self.arm=='counterfactual' else self.alpha)\\n        p = self.parameters()\\n        # Conservative extrapolation: no action inputs outside the trained interval.\\n        effective = (actions*(1+p[:self.a])+self.limit*p[self.a:2*self.a]).clamp(-self.limit,self.limit)\\n        return self.mean(z, effective) + p[2*self.a:]*self.sigma\\n\\n\\ndef calibrate(records, sigma):\\n    \\\"\\\"\\\"Nominal error scale and pooled previous→next residual coefficient.\\n    records: disjoint upstream replan logs with condition and observed_residuals.\\n    Labels used only here; never passed to Correction at test time.\\n    \\\"\\\"\\\"\\n    sigma = torch.as_tensor(sigma, dtype=torch.float64)\\n    nominal, prev, nxt = [], [], []\\n    for r in records:\\n        x = torch.as_tensor(r['observed_residuals'], dtype=torch.float64)/sigma\\n        if r['condition']=='none': nominal.append(x)\\n        prev.append(x[:-1]); nxt.append(x[1:])\\n    if not nominal or not prev:\\n        raise ValueError('disjoint calibration logs required, including nominal episodes')\\n    x, y = torch.cat(prev), torch.cat(nxt)\\n    alpha = ((x*y).sum()/x.square().sum().clamp_min(1e-12)).clamp(0,1)\\n    noise = torch.cat(nominal).square().mean(0).sqrt().clamp_min(.02)\\n    return noise.float(), float(alpha)\\n\\n\\nclass PlannerView:\\n    \\\"\\\"\\\"Expose corrected physical predictions to upstream CEM without modifying its search.\\\"\\\"\\\"\\n    def __init__(self, correction):\\n        self.c = correction\\n        self.a, self.device = correction.m.a, correction.m.device\\n    def predict(self, z, actions):\\n        return self.c.predict(z, actions)\\n    def decode(self, physical):\\n        return physical\\n\",\"run_e25.py\":\"\\\"\\\"\\\"Run the upstream closed-loop benchmark with six correction arms.\\nRequires the original CWM repository with generated model fits, populations and dependencies.\\nCalibration uses disjoint seed_base and trained_predictor only. Test condition is used\\nby the environment, never supplied to the correction. No GitHub execution.\\n\\\"\\\"\\\"\\nimport argparse\\nimport importlib.util\\nimport json\\nfrom pathlib import Path\\nimport sys\\nimport numpy as np\\nimport torch\\nfrom e25 import Correction, PlannerView, calibrate, ARMS\\n\\n\\ndef run_episode(up, sim, m, design, idx, sigma, episode, condition, arm, noise, alpha):\\n    base, steps = design['seed_base'], design['planning_steps']\\n    if condition == 'none':\\n        drivers, goal_driver = np.zeros(steps+2), 0.\\n    elif condition == 'persistent':\\n        g = np.random.default_rng(np.random.SeedSequence([base,episode,11])).standard_normal()\\n        drivers, goal_driver = np.full(steps+2,g),g\\n    else:\\n        drivers = np.random.default_rng(np.random.SeedSequence([base,episode,12])).standard_normal(steps+2)\\n        goal_driver=0.\\n    seed=base+episode\\n    limit=sim.info['action_limit']\\n    goal_command=sim('fix',np.random.default_rng(np.random.SeedSequence([base,episode,14])).uniform(-limit,limit,m.a))\\n    sim('reset',seed); sim('behavior_rng',[base,episode,13]); warm=[]\\n    for t in range(2):\\n        warm.append(sim('behavior',t)); sim('step',warm[-1],drivers[t])\\n    goal=np.asarray(sim('step',goal_command,goal_driver))[idx]\\n    sim('reset',seed)\\n    tensor=lambda x: torch.as_tensor(np.asarray(x),dtype=torch.float32,device=m.device)\\n    c=Correction(m,sigma,limit,noise,alpha,arm)\\n    # Match upstream's render chronology exactly: only the second warm-up\\n    # transition is observed by the model at initialization.\\n    sim('step',warm[0],drivers[0]);zprev=m.encode(sim('render'))\\n    sim('step',warm[1],drivers[1]);z=m.encode(sim('render'))\\n    c.observe(zprev,tensor(warm[1]),m.decode(z))\\n    view=PlannerView(c)\\n    log=dict(episode=episode,condition=condition,arm=arm,distances=[],commands=[],observed_residuals=[])\\n    for t in range(steps):\\n        log['observed_residuals'].append(c.raw_last.cpu().tolist())\\n        generator=torch.Generator(device=m.device).manual_seed(int(base+7919*episode+t))\\n        a=up.cem(view,z,torch.zeros_like(c.sigma),tensor(goal),tensor(sigma),limit,generator,\\n                 design['cem_population'],design['cem_elites'],design['cem_iterations'])\\n        y=np.asarray(sim('step',a.cpu().numpy().astype(np.float64),drivers[t+2]))[idx]\\n        if t+1 < steps:\\n            znew=m.encode(sim('render')); c.observe(z,a,m.decode(znew));z=znew\\n        log['distances'].append(float(np.sqrt(np.mean(((y-goal)/sigma)**2))))\\n        log['commands'].append(a.cpu().tolist())\\n    log['final_distance']=log['distances'][-1]\\n    log['success']=log['final_distance']<design['success_tolerance']\\n    return log\\n\\n\\ndef main():\\n    p=argparse.ArgumentParser(__doc__)\\n    p.add_argument('--upstream',type=Path,required=True)\\n    p.add_argument('--dataset',choices=['hand','cup','fetch','bluerov'],required=True)\\n    p.add_argument('--encoder',choices=['mae_cls','dinov2s_cls'],required=True)\\n    p.add_argument('--seed',type=int,choices=[0,1,2],required=True)\\n    p.add_argument('--device',default='cuda')\\n    p.add_argument('--out',type=Path,required=True)\\n    p.add_argument('--calibration',type=Path,help='Disjoint calibration JSONL; omit to generate calibration only')\\n    p.add_argument('--count',type=int,default=100,help='Official count=100; other values smoke only')\\n    args=p.parse_args();root=args.upstream.resolve();sys.path.insert(0,str(root))\\n    if not 1 <= args.count <= 100: p.error('--count must be in 1..100')\\n    from cwm import registry\\n    from cwm.model import WorldModel\\n    from cwm.sims.worker import Remote\\n    spec=importlib.util.spec_from_file_location('upstream_planning',root/'experiments/4_evaluate/planning.py')\\n    up=importlib.util.module_from_spec(spec);spec.loader.exec_module(up)\\n    design=json.loads((root/'experiments/configs/planning/replan.json').read_text())\\n    original_base=design['seed_base'];d=registry.dataset(args.dataset);cell=design['cells'][args.dataset]\\n    fit=registry.OUT/'fits'/registry.fit_key(dict(dataset=cell['fit'],encoder=args.encoder,history=1,seed=args.seed,recipe='final'))/'fit.pt'\\n    readout=registry.readout(d['readouts'],args.encoder,'state')\\n    for path in (fit,readout,registry.data(d['population'],'population','branches.npz')):\\n        if not path.is_file():raise FileNotFoundError(f'Missing upstream artifact: {path}')\\n    sigma=up.sigma_of(args.dataset)\\n    m=WorldModel(args.encoder,fit,readout,args.device)\\n    tag=dict(dataset=args.dataset,encoder=args.encoder,seed=args.seed)\\n    if args.calibration:\\n        records=[json.loads(line) for line in args.calibration.read_text().splitlines() if line.strip()]\\n        expected=dict(tag,seed_base=original_base+1_000_000,mode='calibration')\\n        if not records or any(any(r.get(k)!=v for k,v in expected.items()) for r in records):\\n            raise ValueError('Wrong calibration cell, mode, or seed base')\\n        keys=[(r['episode'],r['condition'],r['arm']) for r in records]\\n        wanted={(i,c,'trained_predictor') for i in range(100*args.seed,100*args.seed+args.count) for c in design['conditions']}\\n        if set(keys)!=wanted or len(keys)!=len(wanted): raise ValueError('Incomplete or duplicated calibration cohort')\\n        noise,alpha=calibrate(records,sigma);arms=ARMS;mode='test'\\n    else:\\n        design['seed_base']+=1_000_000;noise=np.ones_like(sigma);alpha=0.;arms=['trained_predictor'];mode='calibration'\\n    sim=Remote(registry.simulator(args.dataset,'population'),up.sim_command(args.dataset))\\n    args.out.parent.mkdir(parents=True,exist_ok=True)\\n    try:\\n        with args.out.open('x') as f: # no overwrite or accidental duplicate episodes\\n            for episode in range(100*args.seed,100*args.seed+args.count):\\n                for condition in design['conditions']:\\n                    for arm in arms:\\n                        r=run_episode(up,sim,m,design,d['coordinates'],sigma,episode,condition,arm,noise,alpha)\\n                        r.update(tag,seed_base=design['seed_base'],mode=mode,noise=list(map(float,noise)),alpha=alpha)\\n                        f.write(json.dumps(r)+'\\\\n');f.flush()\\n    finally: sim.close()\\n\\nif __name__=='__main__':main()\\n\",\"score_e25.py\":\"\\\"\\\"\\\"Strict official E25 cohort scorer. No missing cells, no duplicate episodes.\\nUse: python score_e25.py test/*.jsonl --upstream-logs /path/to/outputs/planning/replan\\nWithout upstream logs, scores are provisional (baseline parity not verified).\\n\\\"\\\"\\\"\\nimport argparse\\nimport itertools\\nimport json\\nfrom pathlib import Path\\nimport numpy as np\\n\\nDATASETS=('hand','cup','fetch','bluerov')\\nENCODERS=('mae_cls','dinov2s_cls')\\nCONDITIONS=('persistent','fresh','none')\\nARMS=('trained_predictor','counterfactual','scalar','bias','actuator','joint')\\n\\n\\ndef score(paths, upstream=None, scope=\\\"full\\\"):\\n    cells = [(\\\"hand\\\", \\\"mae_cls\\\", 0)] if scope == \\\"pilot\\\" else list(itertools.product(DATASETS,ENCODERS,range(3)))\\n    expected=set((d,e,s,c,i,a) for d,e,s in cells for c,a in itertools.product(CONDITIONS,ARMS) for i in range(100*s,100*s+100))\\n    rows={}\\n    for path in paths:\\n        for line in Path(path).read_text().splitlines():\\n            r=json.loads(line)\\n            if r['mode']!='test' or r['seed_base']!=3109000000:\\n                raise ValueError('Not a frozen official-test record')\\n            k=tuple(r[v] for v in ('dataset','encoder','seed','condition','episode','arm'))\\n            if k not in expected or k in rows: raise ValueError(f'Unexpected or duplicated {k}')\\n            distances=np.asarray(r['distances'],float)\\n            if distances.shape!=(3,) or not np.all(np.isfinite(distances)) or abs(distances[-1]-r['final_distance'])>1e-9 or bool(r['success'])!=bool(distances[-1]<.2):\\n                raise ValueError(f'Invalid metric {k}')\\n            rows[k]=r\\n    if set(rows)!=expected: raise ValueError(f'Incomplete cohort: {len(rows)}/{len(expected)} rows; no gate score')\\n    # Bootstrap episodes within each dataset/encoder/seed cell. Preserve pairing across conditions and arms.\\n    x=np.array([[[[rows[d,e,s,c,i,a]['success'] for a in ARMS] for c in CONDITIONS] for i in range(100*s,100*s+100)] for d,e,s in cells],dtype=float)\\n    means=x.mean((0,1)); rng=np.random.default_rng(250025)\\n    contrasts=[]\\n    for _ in range(3000):\\n        sample=x[np.arange(len(cells))[:,None],rng.integers(100,size=(len(cells),100))].mean((0,1))\\n        contrasts.append((sample[:,[-1]]-sample[:,:-1]))\\n    b=np.asarray(contrasts); delta=means[:,[-1]]-means[:,:-1]; overall=delta.mean(0)\\n    # Simultaneous one-sided 95% lower bounds over all five overall comparisons.\\n    lower=np.quantile(b.mean(1),.05/5,axis=0)\\n    j=ARMS.index('counterfactual')\\n    parity=None\\n    if upstream:\\n        errors=[]; count=0\\n        for d,e,s in cells:\\n            f=Path(upstream)/f'{d}_{e}_s{s}'/'episodes.jsonl'\\n            found=set()\\n            for line in f.read_text().splitlines():\\n                r=json.loads(line)\\n                if r['task']!='replan' or r['arm'] not in ARMS[:2]:continue\\n                k=(d,e,s,r['condition'],r['episode'],r['arm'])\\n                if k not in rows or k in found: raise ValueError(f'Upstream mismatch {k}')\\n                found.add(k); count+=1\\n                errors.append(abs(r['final_distance']-rows[k]['final_distance']))\\n            if len(found)!=600:raise ValueError(f'Incomplete upstream parity {d,e,s}: {len(found)}')\\n        parity={'n':count,'max_absolute_distance_difference':max(errors),'pass':max(errors)<1e-5}\\n    # Original gate plus comparison to all implemented controls, not a SOTA claim.\\n    gate=(delta[2,j]>=.08 and delta[0,j]>=-.02 and delta[1,j]>=-.02 and overall[j]>=.03 and np.all(lower>0))\\n    return {'n_records':len(rows),'success_percent':{c:{a:100*means[i,j] for j,a in enumerate(ARMS)} for i,c in enumerate(CONDITIONS)},'joint_minus_controls_pp':dict(zip(ARMS[:-1],100*overall)),'simultaneous_lower_pp':dict(zip(ARMS[:-1],100*lower)),'upstream_parity':parity,'implemented_control_gate':bool(gate),'scope':scope,'confirmed_gate':bool(scope=='full' and gate and parity and parity['pass']),'note':'ACTUS comparison and current external SOTA verification still required; no SOTA claim.'}\\n\\nif __name__=='__main__':\\n    p=argparse.ArgumentParser(__doc__);p.add_argument('files',nargs='+');p.add_argument('--upstream-logs',required=True);p.add_argument('--scope',choices=['pilot','full'],default='full');a=p.parse_args()\\n    print(json.dumps(score(a.files,a.upstream_logs,a.scope),indent=2))\\n\",\"test_e25.py\":\"import unittest\\nimport torch\\nfrom e25 import Correction, PlannerView, calibrate, ARMS\\n\\nclass Frozen:\\n    a=2; device='cpu'\\n    def predict(self,z,a):\\n        return a @ z.T\\n    def decode(self,x):\\n        return x\\n\\nclass Tests(unittest.TestCase):\\n    def setUp(self):\\n        self.z=torch.eye(2); self.a=torch.tensor([.3,-.2]); self.m=Frozen()\\n    def make(self, arm): return Correction(self.m,[1,1],1,[.03,.03],.4,arm)\\n    def test_zero_evidence(self):\\n        for arm in ARMS:\\n            torch.testing.assert_close(self.make(arm).predict(self.z,self.a[None]),self.a[None])\\n    def test_zero_residual(self):\\n        for arm in ARMS:\\n            c=self.make(arm);c.observe(self.z,self.a,self.a)\\n            torch.testing.assert_close(c.predict(self.z,self.a[None]),self.a[None])\\n    def test_last_and_scalar(self):\\n        for arm,scale in [('counterfactual',1),('scalar',.4)]:\\n            c=self.make(arm);c.observe(self.z,self.a,self.a+.1)\\n            torch.testing.assert_close(c.predict(self.z,self.a[None]),self.a[None]+scale*.1)\\n    def test_gain_reversal(self):\\n        c=self.make('actuator')\\n        for a in [self.a,-self.a,torch.tensor([.4,.5]),torch.tensor([-.5,-.4])]:\\n            c.observe(self.z,a,a*1.3)\\n        torch.testing.assert_close(c.predict(self.z,self.a[None]),1.3*self.a[None],atol=.007,rtol=0)\\n    def test_joint_identification(self):\\n        c=self.make('joint');b=torch.tensor([.08,-.06])\\n        # Vary J to disambiguate actuator offsets and output bias.\\n        for z in [self.z,self.z*2,self.z*.5]:\\n            for a in [self.a,-self.a,torch.tensor([.5,.4]),torch.tensor([-.4,-.5])]:\\n                c.observe(z,a,(a*1.2+torch.tensor([.04,.03]))@z.T+b)\\n        torch.testing.assert_close(c.predict(self.z,self.a[None]),1.2*self.a[None]+torch.tensor([.04,.03])+b,atol=.005,rtol=0)\\n    def test_boundary_finite(self):\\n        c=self.make('joint');a=torch.ones(2);c.observe(self.z,a,a+.1)\\n        self.assertTrue(torch.isfinite(c.predict(self.z,a[None])).all())\\n    def test_planner_interface(self):\\n        p=PlannerView(self.make('joint'));torch.testing.assert_close(p.decode(p.predict(self.z,self.a[None])),self.a[None])\\n    def test_calibration(self):\\n        noise,alpha=calibrate([dict(condition='none',observed_residuals=[[.1,.1],[.1,.1],[.1,.1]])],[1,1]);self.assertAlmostEqual(alpha,1);torch.testing.assert_close(noise,torch.tensor([.1,.1]))\\n\\nif __name__=='__main__': unittest.main(verbosity=2)\\n\",\"run_suite.py\":\"\\\"\\\"\\\"Run pinned CWM replan cells, then score; no silent skipping or fallback.\\\"\\\"\\\"\\nimport argparse\\nimport itertools\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport subprocess\\nimport sys\\nimport time\\n\\nPIN = '5c84bba92b2ff19d41175070a61238fa1396d10d'\\nHERE = Path(__file__).resolve().parent\\n\\n\\ndef main():\\n    p = argparse.ArgumentParser(__doc__)\\n    p.add_argument('--upstream', type=Path, required=True)\\n    p.add_argument('--work', type=Path, required=True)\\n    p.add_argument('--scope', choices=['pilot', 'full'], required=True)\\n    p.add_argument('--device', default='cuda')\\n    args = p.parse_args()\\n    up, work = args.upstream.resolve(), args.work.resolve()\\n    work.mkdir(parents=True, exist_ok=True)\\n    env = dict(os.environ, CWM_OUTPUTS=str(work/'outputs'), MUJOCO_GL='egl',\\n               PYOPENGL_PLATFORM='egl', SDL_VIDEODRIVER='dummy', WANDB_MODE='disabled')\\n    def run(*cmd, capture=False):\\n        cmd = list(map(str, cmd))\\n        print('>', ' '.join(cmd), flush=True)\\n        return subprocess.run(cmd, cwd=up, env=env, check=True, text=True,\\n                              stdout=subprocess.PIPE if capture else None).stdout\\n    def py(*cmd, capture=False):\\n        return run(sys.executable, *cmd, capture=capture)\\n    if run('git', 'rev-parse', 'HEAD', capture=True).strip() != PIN:\\n        raise RuntimeError('Wrong upstream revision')\\n    py(HERE/'test_e25.py')\\n    cells = [('hand', 'mae_cls', 0)] if args.scope == 'pilot' else list(itertools.product(\\n        ('hand', 'cup', 'fetch', 'bluerov'), ('mae_cls', 'dinov2s_cls'), range(3)))\\n    def index(script, tag, key):\\n        return py(script, tag, '--list', capture=True).splitlines().index(key)\\n    def ready(path, n):\\n        if not path.exists(): return False\\n        try:\\n            rows = [json.loads(x) for x in path.read_text().splitlines()]\\n            if len(rows) == n: return True\\n        except ValueError: pass\\n        path.rename(path.with_suffix(f'.partial.{time.time_ns()}'))\\n        return False\\n    files = []\\n    for d,e,s in cells:\\n        fitd = 'bluerov32k' if d == 'bluerov' else d\\n        py('experiments/1_checkpoints/download.py', e, *(['cup'] if d == 'cup' else []))\\n        for dataset in dict.fromkeys((d, fitd)):\\n            py('experiments/2_data/collect.py', dataset)\\n            py('experiments/2_data/encode.py', dataset, e, '--device', args.device)\\n        py('experiments/2_data/population.py', d)\\n        py('experiments/3_train/readouts.py', d, e, '--index', 0, '--device', args.device)\\n        script = 'experiments/3_train/predictors.py'\\n        py(script, 'planning', '--index', index(script,'planning',f'{fitd}_{e}_h1_s{s}'), '--device', args.device)\\n        script = 'experiments/4_evaluate/planning.py'\\n        py(script, 'replan', '--index', index(script,'replan',f'{d} {e} s{s}'), '--device', args.device)\\n        name = f'{d}_{e}_s{s}.jsonl'\\n        cal, test = work/'calibration'/name, work/'test'/name\\n        base = [HERE/'run_e25.py','--upstream',up,'--dataset',d,'--encoder',e,'--seed',s,'--device',args.device]\\n        if not ready(cal,300): py(*base,'--out',cal)\\n        if not ready(test,1800): py(*base,'--calibration',cal,'--out',test)\\n        files.append(test)\\n    py(HERE/'score_e25.py', *files, '--scope', args.scope, '--upstream-logs', work/'outputs/planning/replan')\\n\\nif __name__ == '__main__': main()\\n\"}")
for name, content in sources.items():
    (BUNDLE/name).write_text(content)
shell(PY,BUNDLE/'test_e25.py')

## Execute and score
This runs the chosen scope end to end, including baseline parity and scoring. The full experiment can require substantial GPU time and disk. Use a fresh directory when changing the implementation.

In [ ]:
shell(PY,BUNDLE/'run_suite.py','--upstream',UP,'--work',ROOT,'--scope',SCOPE,'--device','cuda')

## Export evidence
Source and raw records only; no large weights. If execution failed, this can be run manually to retrieve partial logs.

In [ ]:
from google.colab import files
import zipfile
archive=ROOT/'e25_evidence.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for folder in (BUNDLE,ROOT/'calibration',ROOT/'test',ROOT/'outputs/planning/replan'):
        if folder.exists():
            for f in folder.rglob('*'):
                if f.is_file() and f.suffix in ('.py','.jsonl','.json'):
                    z.write(f,f.relative_to(ROOT))
files.download(str(archive))